# [M4] AI NHÌN VÀ HIỂU DỮ LIỆU
## Buổi S08: Word Embeddings & nn.Embedding Trong PyTorch

---

### Metadata Khóa Học
* **Module:** M4 — AI NHÌN VÀ HIỂU DỮ LIỆU
* **Buổi:** S08
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook thực hành tầng Word Embedding và đo lường khoảng cách Cosine
* **Mức độ liên quan VOAI / Olympic AI:** Nền tảng kiến trúc sâu của mọi mô hình ngôn ngữ từ RNN đến Transformer.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu hạn chế chết người của biểu diễn One-Hot/TF-IDF: Vấn đề trực giao toán học và sự thiếu vắng ngữ nghĩa tương đồng.
- Nắm vững khái niệm Không gian Nhúng Từ (Word Embeddings): Biểu diễn từ bằng các vector thực dày đặc (Dense Vectors) trong không gian d chiều.
- Khám phá trực giác ngữ nghĩa học sâu: Khoảng cách Cosine (Cosine Similarity) và phép tính đại số vector từ vựng (ví dụ: King - Man + Woman ≈ Queen).
- Làm chủ cơ chế hoạt động của tầng `torch.nn.Embedding` trong PyTorch như một Bảng tra cứu tốc độ cao (Lookup Table).

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M4_S06 và M4_S07.
- Hiểu phép nhân tích vô hướng của hai vector (Dot Product).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong cách biểu diễn One-Hot hoặc TF-IDF ở buổi trước, hai từ "khách sạn" và "nhà nghỉ" là hai vector hoàn toàn độc lập và trực giao với nhau trong không gian: Tích vô hướng của chúng bằng 0! Điều này đồng nghĩa với việc máy tính coi khoảng cách giữa "khách sạn" và "nhà nghỉ" cũng xa lạ hệt như khoảng cách giữa "khách sạn" và "quả chuối"!

Để mạng nơ-ron có thể hiểu được ngữ nghĩa sâu sắc, chúng ta cần đưa các từ vào một **Không gian Vectơ Liên tục (Continuous Vector Space)**, nơi các từ có ý nghĩa tương đồng sẽ nằm gần nhau. **Word Embeddings và tầng `nn.Embedding` của PyTorch** chính là trái tim của mọi kiến trúc ngôn ngữ hiện đại, từ RNN, LSTM cho tới những siêu mô hình như GPT-4!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. One-Hot Vector vs Dense Word Embedding
* **One-Hot Encoding:** Vector thưa $V$ chiều (chỉ có duy nhất một số 1, còn lại là số 0). Không mang thông tin ngữ nghĩa, kích thước bùng nổ theo từ điển.
* **Word Embedding:** Vector dày đặc có số chiều cố định $d$ (thường là 64, 128, 300, 768 chiều). Mỗi chiều biểu diễn một thuộc tính tiềm ẩn trừu tượng (ví dụ: giới tính, hoàng gia, động vật...).

### 2. Phép đo Tương đồng Cosine (Cosine Similarity)
Đo góc $\theta$ giữa hai vector trong không gian nhiều chiều:
$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \frac{\sum u_i v_i}{\sqrt{\sum u_i^2} \sqrt{\sum v_i^2}}$$
* Bằng $+1$: Hai từ hoàn toàn đồng nghĩa/cùng hướng.
* Bằng $0$: Hai từ trực giao/không liên quan.
* Bằng $-1$: Hai từ mang ý nghĩa hoàn toàn đối lập.

### 3. Bản chất của tầng `torch.nn.Embedding` trong PyTorch
Tầng `nn.Embedding(num_embeddings, embedding_dim)` thực chất là một **Ma trận Trọng số có thể học được (Learnable Weight Matrix)** kích thước $V \times d$.
* Nó đóng vai trò như một **Bảng tra cứu (Lookup Table)**: Nhận vào chỉ số nguyên (Token ID) và trả về vector $d$ chiều tương ứng trong $O(1)$.
* Các trọng số này được tự động cập nhật qua Backpropagation trong quá trình huấn luyện để tối ưu hóa bài toán!



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Trực quan hóa Không gian Nhúng Từ (Embedding Space 2D):
   Chiều Ngữ nghĩa 2 (Nữ tính)
          ▲
          │          • Queen (Nữ hoàng)
          │         /
          │        /  Vector chuyển dịch [Nữ tính]
          │       /
          │      • King (Vua)
          │
          │                                  • Apple (Quả táo)
          │                                 /
          │                                • Banana (Quả chuối)
          └────────────────────────────────────────────────────────► Chiều Ngữ nghĩa 1 (Hoàng gia)
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Khoi tao tang nn.Embedding trong PyTorch
# Gia su tu dien co 10 tu (vocab_size = 10), moi tu bieu dien bang vector 4 chieu (embedding_dim = 4)
vocab_size = 10
embedding_dim = 4

# padding_idx=0 de vector cua token <PAD> luon duoc giu bang vector 0
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)

print("===== MA TRAN TRONG SO CUA TANG EMBEDDING (10x4) =====")
print(embedding_layer.weight.data)

### 🔍 Code Walkthrough (Khối 1)
Khối code khởi tạo tầng `nn.Embedding`:
- Ma trận trọng số gồm 10 hàng và 4 cột.
- `padding_idx=0`: Đảm bảo token đệm `<PAD>` luôn có vector toàn số 0 và không bị cập nhật gradient.


In [ ]:
# 2. Tra cuu Vector (Lookup Table) va Tinh do tuong dong Cosine
# Gia lap mot cau gom 3 tu co Token IDs: [3, 7, 2]
input_ids = torch.tensor([3, 7, 2], dtype=torch.long)

# Tra cuu vector nhung
word_vectors = embedding_layer(input_ids)
print("===== VECTOR NHUNG CUA CAU [3, 7, 2] =====")
print(f"Kich thuoc tensor vector: {word_vectors.shape} (3 tu, moi tu 4 chieu)")
print(word_vectors)

# 3. Tinh Cosine Similarity giua tu thu nhat (ID=3) va tu thu hai (ID=7)
vec_a = word_vectors[0].unsqueeze(0) # Shape: (1, 4)
vec_b = word_vectors[1].unsqueeze(0) # Shape: (1, 4)

similarity = F.cosine_similarity(vec_a, vec_b)
print(f"\nDo tuong dong Cosine giua Tu 3 va Tu 7: {similarity.item():.4f} (nam trong [-1, 1])")

### 🔍 Code Walkthrough (Khối 2)
Khối code minh họa quá trình tra cứu vector nhúng:
- Truyền tensor các ID nguyên vào tầng embedding thu được tensor 3D đặc trưng.
- Sử dụng `F.cosine_similarity` để đo lường mức độ gần gũi ngữ nghĩa giữa hai vector từ trong không gian nhiều chiều.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Kiểm tra vector của `<PAD>`
Truy xuất vector của token có `ID = 0`. Xác minh xem toàn bộ các phần tử của vector này có bằng chính xác 0.0 hay không.

### Bài 2 (Easy): Khám phá kích thước Batch Embedding
Tạo một batch gồm 2 câu, mỗi câu có 5 từ (tensor kích thước `(2, 5)`). Đưa qua tầng `embedding_layer` ở trên và in ra kích thước của tensor đầu ra. Giải thích ý nghĩa của kích thước `(2, 5, 4)`.

### Bài 3 (Medium): Đại số vector từ vựng (Word Analogy)
Giả sử bạn có các vector Word2Vec của 4 từ: `v_king`, `v_man`, `v_woman`, `v_queen`. Viết công thức tính vector dự đoán $\mathbf{v}_{pred} = \mathbf{v}_{king} - \mathbf{v}_{man} + \mathbf{v}_{woman}$, sau đó tính Cosine Similarity giữa $\mathbf{v}_{pred}$ và $\mathbf{v}_{queen}$.

### Bài 4 (Medium): Nạp trọng số Pre-trained Embeddings vào PyTorch
Tìm hiểu phương thức `nn.Embedding.from_pretrained(pretrained_weights, freeze=True)`. Nêu lợi ích của việc sử dụng các bộ vector từ vựng đã được huấn luyện sẵn (như GloVe hoặc FastText) thay vì khởi tạo ngẫu nhiên từ đầu khi dữ liệu bài thi ít.

### Bài 5 (Challenge): Tính khoảng cách ngữ nghĩa giữa hai văn bản (Sentence Embedding bằng Mean Pooling)
Cho một câu gồm $L$ từ, biểu diễn bằng tensor `(L, embedding_dim)`. Hãy dùng lệnh `torch.mean(..., dim=0)` để tính vector đại diện cho toàn bộ câu (Sentence Embedding). Viết hàm tính độ tương đồng ngữ nghĩa giữa hai câu văn bản bất kỳ bằng phương pháp này.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong các bài thi Olympic AI phần NLP, việc khởi tạo tầng `nn.Embedding` ngẫu nhiên rồi tự huấn luyện từ đầu trên một tập dữ liệu nhỏ (vài nghìn câu) sẽ khiến mô hình học rất chậm và không nắm bắt được ngữ nghĩa phong phú. Thí sinh có tư duy chiến lược sẽ biết cách sử dụng **Pre-trained Embeddings (hoặc ít nhất là khởi tạo cẩn thận kết hợp với Dropout)** để giúp không gian vector từ vựng ổn định ngay từ epoch đầu tiên.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh truyền nhãn mục tiêu hoặc chỉ số từ vựng vượt quá kích thước từ điển vào tầng `nn.Embedding`:

```python
# CODE CÓ LỖI CHỈ SỐ VƯỢT QUÁ KÍCH THƯỚC TỪ ĐIỂN:
# vocab_size = 10 (chi chap nhan chi so tu 0 den 9)
# bad_ids = torch.tensor([4, 15, 2]) # Chi so 15 vuot nguong!
# embedding_layer(bad_ids) # Gay loi: IndexError: index out of range in self!
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao chỉ số Token ID truyền vào `nn.Embedding` bắt buộc phải thỏa mãn $0 \le ID < 	ext{num\_embeddings}$.
2. Nêu cách phòng ngừa lỗi này bằng cơ chế ánh xạ qua `<UNK>` đã học ở Buổi 6.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Ưu điểm vượt trội nhất của Word Embedding (Vector nhúng dày đặc) so với One-Hot Encoding truyền thống là gì?
   - A. Word Embedding chỉ chạy được trên CPU.
   - B. Word Embedding biểu diễn từ bằng các vector số thực liên tục có số chiều nhỏ cố định, trong đó các từ có ngữ nghĩa tương đồng sẽ nằm gần nhau trong không gian.
   - C. Word Embedding không cần dùng bộ nhớ RAM.
   - D. Word Embedding biến văn bản thành hình ảnh.

2. Tầng `nn.Embedding` trong PyTorch về bản chất cấu trúc dữ liệu hoạt động tương tự như thành phần nào?
   - A. Một thuật toán sắp xếp nổi bọt.
   - B. Một Bảng tra cứu ma trận trọng số (Lookup Table), nhận chỉ số nguyên và trả về hàng vector tương ứng.
   - C. Một mạng nơ-ron tích chập 3D.
   - D. Một vòng lặp vô hạn.

3. Giá trị của phép đo Cosine Similarity giữa hai vector luôn nằm trong khoảng nào?
   - A. $[0, +\infty)$
   - B. $[-1, 1]$
   - C. $[0, 255]$
   - D. $(-\infty, +\infty)$

4. Tham số `padding_idx=0` trong `nn.Embedding` có tác dụng gì?
   - A. Xóa bỏ từ đầu tiên của câu.
   - B. Cố định vector tại chỉ số 0 luôn bằng 0 và không cập nhật gradient cho vị trí này trong quá trình huấn luyện.
   - C. Tăng kích thước từ điển lên gấp đôi.
   - D. Đảo ngược thứ tự các từ.

5. Trong không gian Word2Vec kinh điển, phép toán trừu tượng nào sau đây thể hiện mối quan hệ ngữ nghĩa tương tự?
   - A. $	ext{Vector}("Vua") + 	ext{Vector}("Táo") = 	ext{Vector}("Cam")$
   - B. $	ext{Vector}("King") - 	ext{Vector}("Man") + 	ext{Vector}("Woman") pprox 	ext{Vector}("Queen")$
   - C. $	ext{Vector}("Xe") 	imes 	ext{Vector}("Nhà") = 0$
   - D. $	ext{Vector}("Chó") - 	ext{Vector}("Mèo") = 1$



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Hạn chế trực giao của One-Hot và sức mạnh của Không gian nhúng từ (Word Embeddings).
  - Cách tính toán và ý nghĩa của khoảng cách tương đồng Cosine.
  - Cơ chế Bảng tra cứu trọng số của tầng `torch.nn.Embedding`.
* **What I can now do:**
  - Tích hợp tầng `nn.Embedding` vào bất kỳ kiến trúc mạng nơ-ron PyTorch nào.
  - Đo lường mức độ tương đồng ngữ nghĩa giữa các vector từ vựng trong không gian nhiều chiều.
* **What I still need to practice:**
  - Lắp ráp tầng Embedding vào một **Text Classification Pipeline hoàn chỉnh với PyTorch** ở Buổi 9.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M4_S09**, chúng ta sẽ xây dựng cỗ máy học sâu ngôn ngữ hoàn chỉnh: **Xây Dựng Text Classification Pipeline Hoàn Chỉnh với PyTorch** — Tự viết Custom Collate Function cho các câu có độ dài biến thiên, kết hợp Embedding với Pooling và huấn luyện phân loại cảm xúc văn bản!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M4**:
- Bản chất Word Embeddings & Không gian vector liên tục ✓
- Đo khoảng cách ngữ nghĩa bằng Cosine Similarity ✓
- Tầng nn.Embedding trong PyTorch như một Lookup table ✓

* **Session Deliverable:** Mã nguồn tra cứu vector nhúng và tính khoảng cách ngữ nghĩa bằng PyTorch
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
